# 03. Functions, closures and decorators

## Learning goals
1. Apply the LEGB scope rule.
2. Explain how closures capture variables in cells.
3. Avoid the late-binding trap.
4. Write decorators and decorator factories that keep metadata.
5. Prove why memoisation turns exponential recursion into linear time.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## Scope: LEGB
Python looks up a name in Local, Enclosing, Global, then Built-in scopes. Assignment makes a name local unless declared `global` or `nonlocal`.

In [ ]:
x = "global"
def outer():
    x = "enclosing"
    def inner():
        return x
    return inner()
assert outer() == "enclosing"
def shadow():
    try:
        print(y)
    except UnboundLocalError:
        return "unbound"
    y = 1
assert shadow() == "unbound"
assert len([1, 2]) == 2

## Closures
An inner function keeps the variables it uses in cells. The cells live as long as the function does.

In [ ]:
from understanding_python.functional import make_counter
a, b = make_counter(), make_counter(10)
assert [a(), a(), b()] == [1, 2, 11]
print("free vars:", a.__code__.co_freevars)
assert a.__code__.co_freevars == ("count",)
assert a.__closure__[0].cell_contents == 2

## Late binding
A closure sees the variable, not its value at creation time. Bind the value with a default argument.

In [ ]:
late = [lambda: i for i in range(3)]
early = [lambda i=i: i for i in range(3)]
assert [f() for f in late] == [2, 2, 2]
assert [f() for f in early] == [0, 1, 2]

## Decorators
`@d` above `def f` means `f = d(f)`. Use `functools.wraps` to keep the name and docstring.

In [ ]:
import functools
from understanding_python.functional import memoize, repeat
def logged(func):
    calls = []
    @functools.wraps(func)
    def wrapper(*a, **k):
        calls.append(a)
        return func(*a, **k)
    wrapper.calls = calls
    return wrapper
@logged
def add(x, y):
    "Add two numbers."
    return x + y
assert add(1, 2) == 3 and add.calls == [(1, 2)]
assert add.__name__ == "add" and add.__doc__ == "Add two numbers."
@repeat(3)
def ping(): return "p"
assert ping() == ["p", "p", "p"]

## Complexity: memoised Fibonacci
**Naive recursion.** T(n) = T(n-1) + T(n-2) + 1. By induction T(n) >= F(n), and F(n) grows like phi^n with phi near 1.618, so the cost is exponential.

**Memoised.** Each distinct argument from 0 to n computes once. Every computation does O(1) work beyond cached calls.
There are n+1 distinct arguments, so the cost is O(n) time and O(n) space.

In [ ]:
from understanding_python.dp import fib_naive, fib_table
from understanding_python.bench import best_time
@memoize
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
assert [fib(i) for i in range(10)] == [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
assert fib(80) == fib_table(80) == 23416728348467685
assert len(fib.cache) == 81
t_naive = best_time(lambda: fib_naive(20), 1)
t_memo = best_time(lambda: fib_table(20), 1)
print(f"naive {t_naive:.5f}s  linear {t_memo:.7f}s")
assert t_memo < t_naive

## Functions are objects
You can store them, pass them and inspect them. `functools.partial` fixes arguments.

In [ ]:
from functools import partial, reduce
power = lambda base, exp: base ** exp
square = partial(power, exp=2)
assert square(7) == 49
assert reduce(lambda acc, f: f(acc), [square, lambda n: n + 1], 3) == 10
assert sorted(["bb", "a", "ccc"], key=len, reverse=True) == ["ccc", "bb", "a"]
def f(a, b=2, *args, c, **kw): pass
import inspect
assert list(inspect.signature(f).parameters) == ["a", "b", "args", "c", "kw"]